In [ ]:
import _plotting as plot
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from xxm.core.affine import Affine
from xxm.core.align import match_states_by_conditional_mean as match_states
from xxm.hmm import PoissonARHMM

In [ ]:
NUM_NEURONS = 10
NUM_TIME_STEPS = 5000


def make_true_model_bimodal(
    num_neurons,
):
    baseline = jnp.full(num_neurons, 8.0)

    # Deterministic 2D population subspace.
    angles = jnp.linspace(
        0,
        2 * jnp.pi,
        num_neurons,
        endpoint=False,
    )

    basis = jnp.stack(
        [
            jnp.cos(angles),
            jnp.sin(angles),
        ],
        axis=1,
    )

    basis, _ = jnp.linalg.qr(basis)  # (N, 2)

    local_dynamics_2d = jnp.array(
        [
            # clockwise
            [
                [0.60, 0.25],
                [-0.25, 0.60],
            ],
            # counter-clockwise
            [
                [0.60, -0.25],
                [0.25, 0.60],
            ],
        ]
    )

    projection = basis @ basis.T
    orthogonal_projection = jnp.eye(num_neurons) - projection

    local_dynamics = jax.vmap(
        lambda dynamics: basis @ dynamics @ basis.T + 0.2 * orthogonal_projection
    )(local_dynamics_2d)  # (K, N, N)

    coefficients = (local_dynamics / baseline[:, None])[
        :, :, None, :
    ]  # (K, O=N, L=1, I=N)

    biases = jnp.log(baseline)[None, :] - jnp.einsum(
        'koi,i->ko',
        coefficients[:, :, 0, :],
        baseline,
    )

    model = PoissonARHMM.from_params(
        initial_probs=jnp.array([0.4, 0.6]),
        transition_probs=jnp.array(
            [
                [0.98, 0.02],
                [0.02, 0.98],
            ]
        ),
        emission_coefficients=coefficients,
        emission_bias=biases,
    )

    embedding = Affine(
        coefficients=basis,
        bias=baseline,
    )

    return model, embedding


def make_true_model_monomodal(
    num_neurons,
):
    baseline = jnp.full(num_neurons, 8.0)

    # Deterministic 2D population subspace.
    angles = jnp.linspace(
        0,
        2 * jnp.pi,
        num_neurons,
        endpoint=False,
    )

    basis = jnp.stack(
        [
            jnp.cos(angles),
            jnp.sin(angles),
        ],
        axis=1,
    )

    basis, _ = jnp.linalg.qr(basis)  # (N, 2)

    local_dynamics_2d = jnp.array(
        [
            # clockwise
            [
                [0.60, 0.25],
                [-0.25, 0.60],
            ],
        ]
    )

    projection = basis @ basis.T
    orthogonal_projection = jnp.eye(num_neurons) - projection

    local_dynamics = jax.vmap(
        lambda dynamics: basis @ dynamics @ basis.T + 0.2 * orthogonal_projection
    )(local_dynamics_2d)  # (K, N, N)

    coefficients = (local_dynamics / baseline[:, None])[
        :, :, None, :
    ]  # (K, O=N, L=1, I=N)

    biases = jnp.log(baseline)[None, :] - jnp.einsum(
        'koi,i->ko',
        coefficients[:, :, 0, :],
        baseline,
    )

    model = PoissonARHMM.from_params(
        initial_probs=jnp.array([1.0]),
        transition_probs=jnp.array(
            [
                [1.0],
            ]
        ),
        emission_coefficients=coefficients,
        emission_bias=biases,
    )

    embedding = Affine(
        coefficients=basis,
        bias=baseline,
    )

    return model, embedding


true_model, embedding = make_true_model_bimodal(num_neurons=NUM_NEURONS)

true_states, observations = true_model.sample(
    num_steps=NUM_TIME_STEPS,
    key=jax.random.key(0),
)

In [ ]:
def plot_data(true_states, observations):

    _, axs = plt.subplots(
        nrows=2,
        sharex='all',
        figsize=(8, 3),
        constrained_layout=True,
        gridspec_kw={'height_ratios': [1, 2]},
    )

    ax = axs[0]

    plot.plot_state_1d(ax, true_states)

    ax = axs[1]
    plot.plot_traces_image(ax, observations, ylabel='neuron')


plot_data(true_states, observations)

In [ ]:
def plot_state_dynamics(ax, model, embedding, state, **kwargs):
    projection = embedding.pseudoinverse()
    dynamics = (
        model.states.select(state)
        .reshape_input((embedding.output_dim,))  # drop lag single dim
        .compose_input(embedding)
    )

    plot.plot_dyn_2d(
        ax,
        lambda z: projection.apply(dynamics.conditional(z).rates),
        **kwargs,
    )


def plot_model_dynamics(model: PoissonARHMM, embedding: Affine) -> None:
    """Plot the expected displacement field of a 2D linear dynamics model."""

    _, axs = plt.subplots(
        ncols=model.num_states,
        figsize=(8, 3),
        sharex=True,
        sharey=True,
        constrained_layout=True,
        squeeze=False,
    )

    axs = axs.ravel()

    for state, ax in enumerate(axs):
        plot_state_dynamics(ax, model, embedding, state)

        ax.set_title(f'State {state}')


plot_model_dynamics(true_model, embedding)

In [ ]:
posterior, _ = true_model.infer(observations)

In [ ]:
def plot_inference_comparison(
    true_states, observations, inferred_states, reconstructions
):
    _f, axs = plt.subplots(
        nrows=4,
        constrained_layout=True,
        figsize=(8, 8),
        gridspec_kw={'height_ratios': [1, 2, 1, 2]},
    )

    ax = axs[0]
    plot.plot_state_1d(ax, true_states)
    ax.set(title='True states')

    ax = axs[1]
    plot.plot_traces_image(ax, reconstructions, ylabel='neuron')
    ax.set(title='Reconstructions')

    ax = axs[2]

    inferred_states = posterior.state_probs.argmax(axis=1)
    plot.plot_state_1d(ax, inferred_states)
    ax.set(title='Inferred states')

    ax = axs[3]
    plot.plot_traces_image(ax, observations, ylabel='neuron')
    ax.set(title='Observations')


plot_inference_comparison(
    true_states=true_states,
    observations=observations,
    inferred_states=true_model.most_likely_states(posterior),
    reconstructions=true_model.observation_mean(observations, posterior),
)

In [ ]:
initial_model = PoissonARHMM.from_kmeans(
    observations=observations,
    num_states=true_model.num_states,
    key=jax.random.key(0),
    num_lags=1,
)


fit = initial_model.fit(
    observations=observations,
    num_iters=100,
)


plot.plot_fit_progress(fit.objective_trace)

In [ ]:
def align_model(learned_model, true_model):

    permutation = match_states(
        learned_model.states_conditional(observations).rates,
        true_model.states_conditional(observations).rates,
    )
    learned_model = learned_model.permute(permutation)

    return learned_model


learned_model = align_model(fit.model, true_model)

In [ ]:
def plot_model_dynamics_comparison(
    model0: PoissonARHMM, model1: PoissonARHMM, embedding: Affine
) -> None:
    """Plot the expected displacement field of a 2D linear dynamics model."""

    _, axs = plt.subplots(
        ncols=model0.num_states,
        figsize=(8, 3),
        sharex=True,
        sharey=True,
        constrained_layout=True,
        squeeze=False,
    )

    axs = axs.ravel()

    for state, ax in enumerate(axs):
        plot_state_dynamics(ax, model0, embedding, state, color='k')
        plot_state_dynamics(ax, model1, embedding, state, color='xkcd:magenta')

        ax.set_title(f'State {state}')


plot_model_dynamics_comparison(true_model, learned_model, embedding)

In [ ]:
posterior_learnt, _ = learned_model.infer(observations)

In [ ]:
plot_inference_comparison(
    true_states=true_states,
    observations=observations,
    inferred_states=learned_model.most_likely_states(posterior_learnt),
    reconstructions=learned_model.observation_mean(observations, posterior_learnt),
)